# Modeling

Training CNN architectures on the augmented OCT volume dataset.

**Key design choices:**
- Train on augmented data (6x original) with normalization (÷255) and shuffling
- Evaluate on the original (non-augmented) test set for fair comparison with Part 1

## Imports

In [ ]:
import sys
sys.path.append('..')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from octcv.mdl_lib import yX_split, XVolSet, ModelEvaluator, train_model
from octcv.mdl_lib.architectures import buildSequential, buildResNet, buildAttnNN
from octcv.mdl_lib.callbacks import LivePlot
from octcv.system_monitoring import startMemoryMonitor

In [ ]:
startMemoryMonitor()

## Load & Split Data

In [ ]:
# Load augmented metadata (6660 entries: 1110 original + 5×1110 augmented)
aug_df = pd.read_csv('../datasrc/volumeOCT-AUGMENTED_metadata.csv')
print(f"Augmented dataset: {len(aug_df)} entries")
print(f"Augmentations: {aug_df['augmentation'].unique()}")

# Load original train/test splits from Part 1
v_train = pd.read_csv('../p5_Modeling/volumes_train.csv')
v_test = pd.read_csv('../p5_Modeling/volumes_TEST.csv')
print(f"\nOriginal split: {len(v_train)} train / {len(v_test)} test")

In [ ]:
# Build augmented TRAINING set (all 6 versions of each training volume)
nonpath_columns = ['dx_class', 'glaucoma', 'PIN', 'laterality', 'left_eye', 'set']
augmentations = aug_df['augmentation'].unique()

augpiv = aug_df.pivot(
    index=nonpath_columns, columns='augmentation', values='display_volume'
).rename_axis(None, axis=1).reset_index()

augpiv_train = pd.merge(
    v_train, augpiv[list(augmentations)],
    left_on='display_volume', right_on='original', how='left'
).drop(columns='display_volume')

augTRAIN = pd.melt(
    augpiv_train, id_vars=nonpath_columns,
    var_name='augmentation', value_vars=augmentations,
    value_name='display_volume'
)

# VALIDATION: use augmented versions for richer validation signal during training
# EVALUATION: use ONLY original test volumes (fair comparison with Part 1)
v_valid, v_eval = train_test_split(v_test, test_size=0.5, random_state=42)

print(f"Augmented train: {len(augTRAIN)}")
print(f"Validation (original): {len(v_valid)}")
print(f"Evaluation (original): {len(v_eval)}")

## Create tf.data.Dataset Pipelines

Volumes are loaded lazily in batches, normalized to [0,1], and shuffled.

In [ ]:
BATCH_SIZE = 4

# Training: augmented, shuffled, normalized
_, ay_train, aX_train = yX_split(augTRAIN, display_filepath_columns='display_volume')

# Validation: original volumes only, shuffled
_, vy_valid, vX_valid = yX_split(v_valid)

# Evaluation: original volumes only, no shuffle (deterministic)
_, vy_eval, vX_eval = yX_split(v_eval)

print(f"Input shape: {aX_train.input_shape}")
print(f"Train batches/epoch: ~{len(aX_train) // BATCH_SIZE}")
print(f"Validation samples: {len(vX_valid)}")
print(f"Evaluation samples: {len(vX_eval)}")

## Model 1: Sequential CNN

In [ ]:
model_seq = buildSequential()
model_seq.summary()

In [ ]:
train_ds = aX_train.to_tf_dataset(ay_train, batch_size=BATCH_SIZE, shuffle=True)
valid_ds = vX_valid.to_tf_dataset(vy_valid, batch_size=BATCH_SIZE, shuffle=False)

history_seq = train_model(
    model_seq, train_ds, valid_ds,
    epochs=50, patience=5, learning_rate=1e-4,
    live_plot=True, metrics_to_plot=['auc', 'val_auc']
)

In [ ]:
eval_ds = vX_eval.to_tf_dataset(vy_eval, batch_size=BATCH_SIZE, shuffle=False)
evaluator_seq = ModelEvaluator(model_seq, history_seq, eval_ds)
evaluator_seq.summary()

## Model 2: ResNet-Like

In [ ]:
model_res = buildResNet()

train_ds = aX_train.to_tf_dataset(ay_train, batch_size=BATCH_SIZE, shuffle=True)
valid_ds = vX_valid.to_tf_dataset(vy_valid, batch_size=BATCH_SIZE, shuffle=False)

history_res = train_model(
    model_res, train_ds, valid_ds,
    epochs=50, patience=5, learning_rate=1e-4,
    live_plot=True
)

In [ ]:
eval_ds = vX_eval.to_tf_dataset(vy_eval, batch_size=BATCH_SIZE, shuffle=False)
evaluator_res = ModelEvaluator(model_res, history_res, eval_ds)
evaluator_res.summary()

## Model 3: Attention Network

In [ ]:
model_attn = buildAttnNN()

train_ds = aX_train.to_tf_dataset(ay_train, batch_size=BATCH_SIZE, shuffle=True)
valid_ds = vX_valid.to_tf_dataset(vy_valid, batch_size=BATCH_SIZE, shuffle=False)

history_attn = train_model(
    model_attn, train_ds, valid_ds,
    epochs=50, patience=5, learning_rate=1e-4,
    live_plot=True
)

In [ ]:
eval_ds = vX_eval.to_tf_dataset(vy_eval, batch_size=BATCH_SIZE, shuffle=False)
evaluator_attn = ModelEvaluator(model_attn, history_attn, eval_ds)
evaluator_attn.summary()

## Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
models_data = [
    (evaluator_seq, 'Sequential'),
    (evaluator_res, 'ResNet-Like'),
    (evaluator_attn, 'Attention'),
]

for ax, (ev, name) in zip(axes, models_data):
    ev.plot_roc(ax=ax)
    ax.set_title(f'{name}\nAUC = {ev.roc_auc():.4f}')

plt.suptitle('ROC Comparison - Augmented Training, Original Eval', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\n{'Model':<15} {'AUC':<8} {'Threshold':<12}")
print('-' * 35)
for ev, name in models_data:
    print(f"{name:<15} {ev.roc_auc():.4f}   {ev.threshold:.4f}")

print(f"\nPart 1 baselines: Sequential ~0.88, ResNet ~0.94, Attention ~0.93")

## Hyperparameter Tuning

Grid search over learning rates using the best-performing architecture from above.

In [ ]:
learning_rates = [5e-5, 1e-4, 3e-4, 5e-4]
results = []

for lr in learning_rates:
    print(f"\nLR = {lr}")
    model = buildSequential(model_name=f'seq_lr{lr}')

    train_ds = aX_train.to_tf_dataset(ay_train, batch_size=BATCH_SIZE, shuffle=True)
    valid_ds = vX_valid.to_tf_dataset(vy_valid, batch_size=BATCH_SIZE, shuffle=False)

    history = train_model(
        model, train_ds, valid_ds,
        epochs=30, patience=4, learning_rate=lr, verbose=2
    )

    eval_ds = vX_eval.to_tf_dataset(vy_eval, batch_size=BATCH_SIZE, shuffle=False)
    ev = ModelEvaluator(model, history, eval_ds)
    auc = ev.roc_auc()
    results.append({'lr': lr, 'auc': auc, 'epochs': len(history.history['loss'])})
    print(f"  AUC: {auc:.4f} in {results[-1]['epochs']} epochs")

In [ ]:
results_df = pd.DataFrame(results).sort_values('auc', ascending=False)
print("Hyperparameter Tuning Results:")
print(results_df.to_string(index=False))

best_lr = results_df.iloc[0]['lr']
print(f"\nBest learning rate: {best_lr}")

## Final Training

In [ ]:
final_model = buildSequential(model_name='final_augmented')

train_ds = aX_train.to_tf_dataset(ay_train, batch_size=BATCH_SIZE, shuffle=True)
valid_ds = vX_valid.to_tf_dataset(vy_valid, batch_size=BATCH_SIZE, shuffle=False)

final_history = train_model(
    final_model, train_ds, valid_ds,
    epochs=80, patience=8, learning_rate=best_lr,
    live_plot=True
)

In [ ]:
eval_ds = vX_eval.to_tf_dataset(vy_eval, batch_size=BATCH_SIZE, shuffle=False)
final_evaluator = ModelEvaluator(final_model, final_history, eval_ds)
final_evaluator.summary()

## Save Model

In [ ]:
save_dir = os.path.join('..', 'p5_Modeling', 'models', 'augmented_final')
os.makedirs(save_dir, exist_ok=True)
final_model.save(os.path.join(save_dir, 'augmented_final.keras'))
print(f"Model saved to: {save_dir}")